<a href="https://colab.research.google.com/github/raven-in-space/data-science-cohort-20/blob/main/Project-2/Predicting_Housing_Prices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 2: Housing Price Prediction


The project should be done using Regression prediction techniques ( with feature scaling and regularization ) as covered in class.
The goal is to minimize the RMS ***percentage*** error ( root mean squared percentage error - RMSPE ) on your prediction of the house sales price.



Any of the available features can be used in the regression, but a minimum set of variables that do the job should be the ultimate goal.



Be sure to go through the whole data science process and document as such in your Jupyter notebook.



This project will have less direct "To Do" guidance and the progression through the data science process will be more individualized this time around.
We will talk about all the issues during class so you're not going to be out on a ledge with this one, I just want to move you towards performing a data science project on your own eventually.



A data dictionary file is available at AWS S3 at [Housing Data Dictionary]( https://ddc-datascience.s3.amazonaws.com/Projects/Project.2-Housing/Housing%20-%20Data%20Documentation.pdf ).

The data is available on AWS S3 at https://ddc-datascience.s3.amazonaws.com/Projects/Project.2-Housing/Data/Housing.Data.csv .


# Predicting Housing Prices Using Linear Regression and RMSPE
* **Date <u>Started</u>:** `4/10/2026`
* **Author:** Raven Otero-Symphony
* **Program:** CNM Ingenuity [Data Science Bootcamp]()

# Problem Definition

The Accessor's Office has provided data on individual residential properties sold from 2006 to 2010. This project aims to *reliably* predict house sales prices through feature scaling, regularization, and minimixing the **Root Mean Square *Percentage* Error (RMSPE)** in a predictive regression model. Secondarily, we aim to find the most reliable model with the *least* amount of variables possible to predict housing prices.

# Data Collection

We are using Python data cleaning and regression methods, including `sklearn`, to fit and evaluate our predictive model.

**Data Sources:**
1. [Source Data](https://ddc-datascience.s3.amazonaws.com/Projects/Project.2-Housing/Data/Housing.Data.csv) - collected from AWS S3.
2. [Data Dictionary](https://ddc-datascience.s3.amazonaws.com/Projects/Project.2-Housing/Housing%20-%20Data%20Documentation.pdf) - a brief description of the data and its and features.

🎈 Stands for comments related to future development.

# Data Cleaning

## Load

In [ ]:
import pandas as pd
import numpy as np

## IDE

In [ ]:
orig_housing = pd.read_csv("https://ddc-datascience.s3.amazonaws.com/Projects/Project.2-Housing/Data/Housing.Data.csv")
orig_housing.shape

(2637, 81)

In [ ]:
orig_housing.head(5)

,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,905101070,20,RL,62.0,14299,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,7,2007,WD,Normal,115400
1,905101330,90,RL,72.0,10791,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,Shed,500,10,2006,WD,Normal,90000
2,903454090,50,RM,50.0,9000,Pave,NaN,Reg,Bnk,AllPub,...,0,NaN,NaN,NaN,0,12,2007,WD,Normal,141000
3,533244030,60,FV,68.0,7379,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,254000
4,909252020,70,RL,60.0,7200,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,4,2009,WD,Normal,155000


In [ ]:
orig_housing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2637 entries, 0 to 2636
Data columns (total 81 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   PID              2637 non-null   int64  
 1   MS SubClass      2637 non-null   int64  
 2   MS Zoning        2637 non-null   object 
 3   Lot Frontage     2188 non-null   float64
 4   Lot Area         2637 non-null   int64  
 5   Street           2637 non-null   object 
 6   Alley            180 non-null    object 
 7   Lot Shape        2637 non-null   object 
 8   Land Contour     2637 non-null   object 
 9   Utilities        2637 non-null   object 
 10  Lot Config       2637 non-null   object 
 11  Land Slope       2637 non-null   object 
 12  Neighborhood     2637 non-null   object 
 13  Condition 1      2637 non-null   object 
 14  Condition 2      2637 non-null   object 
 15  Bldg Type        2637 non-null   object 
 16  House Style      2637 non-null   object 
 17  Overall Qual  

**Initial Observations:**
* According to the [data dictionary](https://ddc-datascience.s3.amazonaws.com/Projects/Project.2-Housing/Housing%20-%20Data%20Documentation.pdf), there should be **2,930 observations (rows)** and **82 variables (features)**. What we find instead is that there are **2,637 rows** and **81 columns**. That's a difference of **293 rows** and **1 column**. I wonder which feature has the most missing rows, and why?
* The `MS Subclass` feature is being picked up as an integer rather than the nominal variable it actually is. Type conversions will be necessary for some features!
* The classification of nominal versus ordinal variables is interesting within the dictionary. For example, `Lot Shape` is considered ordinal but `Land Contour` is considered nominal.
* The `Sawyer` versus `Sawyer West` naming in the `Neighborhood` feature is *hopefully* labeled correctly in the actual data... hopefully (re-classifying would require a judgement call since I can't actually reach out to the PoC).
* I might combine or drop `Condition 1` with `Condition 2`, `Exterior 1` with `Exterior 2`, `Exter Qual` with `Exter Cond`, `Bsmt Qual` with `Bsmt Cond`, depending on correlation strength and model relevancy, since the labels within these features repeat themselves.
* As an aside, it would be interesting to look at the `Overall Qual` and `Overall Cond` features (since those seem self-reported) against other features of the house. Did the surveyor assume quality in a consistent fashion across houses? Or did they hold some biases?
* `SalePrice` is the target variable and has no missing rows.

In [ ]:
# create a copy
main_housing = orig_housing.copy()

# create filters for relevant summary stats
stats_cat = main_housing.select_dtypes(include='object') # "cat" stands for "categorical"
stats_num = main_housing.select_dtypes(include='number') # "num" stands for "numerical"

# visualize the categorical features
stats_cat.describe().transpose().sort_values(by='freq', ascending=False)

,count,unique,top,freq
Utilities,2637,3,AllPub,2634
Street,2637,2,Pave,2625
Condition 2,2637,8,Norm,2609
Roof Matl,2637,8,CompShg,2599
Heating,2637,6,GasA,2597
Land Slope,2637,3,Gtl,2511
Central Air,2637,2,Y,2460
Functional,2637,8,Typ,2453
Electrical,2637,5,SBrkr,2414
Garage Cond,2490,5,TA,2400


In [ ]:
# now the numeric features
stats_num.describe().round(1).transpose().sort_values(by='count', ascending=False)

,count,mean,std,min,25%,50%,75%,max
PID,2637.0,714130147.7,188752674.8,526301100.0,528477010.0,535453040.0,907187010.0,1.007100e+09
MS SubClass,2637.0,57.3,42.5,20.0,20.0,50.0,70.0,1.900000e+02
Lot Area,2637.0,10044.7,6742.5,1300.0,7436.0,9450.0,11526.0,1.646600e+05
Overall Qual,2637.0,6.1,1.4,1.0,5.0,6.0,7.0,1.000000e+01
Overall Cond,2637.0,5.6,1.1,1.0,5.0,5.0,6.0,9.000000e+00
Year Built,2637.0,1971.3,30.3,1872.0,1954.0,1973.0,2001.0,2.010000e+03
Misc Val,2637.0,42.0,393.2,0.0,0.0,0.0,0.0,1.250000e+04
Year Remod/Add,2637.0,1984.2,20.9,1950.0,1965.0,1993.0,2004.0,2.010000e+03
1st Flr SF,2637.0,1155.5,382.6,334.0,878.0,1082.0,1380.0,4.692000e+03
Bedroom AbvGr,2637.0,2.9,0.8,0.0,2.0,3.0,3.0,6.000000e+00


In [ ]:
# Create a summary table of missing values
null_summary = pd.DataFrame({
    'Null Count': main_housing.isna().sum(),
    'Percentage': (main_housing.isna().sum() / len(main_housing)) * 100
}).sort_values(by='Null Count', ascending=False)

null_summary

,Null Count,Percentage
Pool QC,2626,99.582859
Misc Feature,2541,96.359499
Alley,2457,93.174061
Fence,2109,79.977247
Mas Vnr Type,1607,60.940463
...,...,...
Mo Sold,0,0.000000
Yr Sold,0,0.000000
Sale Type,0,0.000000
Sale Condition,0,0.000000


**Missing Data Summary:**
* **5 features** have **over 50%** of missing data. Although 48% of `Fireplace Qu` is missing, I'm going to drop that feature as well because it is so close to 50% in context of the rest of the data.
* **10 features** are missing **2-5%** of data and **1 feature** is missing **17%** of data. These can be imputed, although my hunch is that some of the features will be highly correlated amongst themselves (do data scientists work off of hunches? I do).
* **9 features** are missing less than **1%** of data. Again, this can be imputed if necessary.
* The remaining **57 features** aren't missing any data, including the target variable.

In [ ]:
# quick math from total values minus rounded cutoff Null Count (147+3=150)
2636-150

2486

In [ ]:
# checkpoint
housing_not_cleaned = main_housing.copy()

# drop features at cutoff
cutoff = ( len(main_housing) * 0.3 )
main_housing = main_housing.dropna(axis = 1, thresh=cutoff)
main_housing.columns

# 🎈 A more graceful way to list columns dropped? tried merge and .isnull() but didn't like the results

Index(['PID', 'MS SubClass', 'MS Zoning', 'Lot Area', 'Street', 'Lot Shape',
       'Land Contour', 'Utilities', 'Lot Config', 'Land Slope', 'Neighborhood',
       'Condition 1', 'Condition 2', 'Bldg Type', 'House Style',
       'Overall Qual', 'Overall Cond', 'Year Built', 'Year Remod/Add',
       'Roof Style', 'Roof Matl', 'Exterior 1st', 'Exterior 2nd',
       'Mas Vnr Area', 'Exter Qual', 'Exter Cond', 'Foundation', 'Bsmt Qual',
       'Bsmt Cond', 'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin SF 1',
       'BsmtFin Type 2', 'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF',
       'Heating', 'Heating QC', 'Central Air', 'Electrical', '1st Flr SF',
       '2nd Flr SF', 'Low Qual Fin SF', 'Gr Liv Area', 'Bsmt Full Bath',
       'Bsmt Half Bath', 'Full Bath', 'Half Bath', 'Bedroom AbvGr',
       'Kitchen AbvGr', 'Kitchen Qual', 'TotRms AbvGrd', 'Functional',
       'Fireplaces', 'Garage Type', 'Garage Yr Blt', 'Garage Finish',
       'Garage Cars', 'Garage Area', 'Garage Qual', 'Garage Co

In [ ]:
main_housing["Street"].value_counts( dropna= False)

,count
Street,
Pave,2625
Grvl,12


* Strange how `Condition 2` appears more than `Condition 1` when the data dictionary states that `Condition 2` exists only if more than one characteristic from `Condition 1` exists...

In [ ]:
stats_cat.head()

,MS Zoning,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,Land Slope,Neighborhood,Condition 1,...,Garage Type,Garage Finish,Garage Qual,Garage Cond,Paved Drive,Pool QC,Fence,Misc Feature,Sale Type,Sale Condition
0,RL,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,Sawyer,Feedr,...,Detchd,Unf,TA,TA,N,NaN,MnPrv,NaN,WD,Normal
1,RL,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,Sawyer,Norm,...,CarPort,Unf,TA,TA,Y,NaN,NaN,Shed,WD,Normal
2,RM,Pave,NaN,Reg,Bnk,AllPub,Inside,Gtl,IDOTRR,Norm,...,Detchd,Unf,TA,TA,P,NaN,NaN,NaN,WD,Normal
3,FV,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,Somerst,Norm,...,Attchd,RFn,TA,TA,Y,NaN,NaN,NaN,WD,Normal
4,RL,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,SWISU,Feedr,...,Detchd,RFn,TA,TA,Y,NaN,MnPrv,NaN,WD,Normal


In [ ]:
main_housing['Pool QC'].unique()

array([nan, 'Gd', 'Fa', 'TA', 'Ex'], dtype=object)

# Exploratory Data Analysis

* Produce some visual analysis of the data – like plots showing the distributions of all variables. Recall that Gaussian Naive Bayes assumes the predictors are normally distributed. Note: you might have to do multiple plots in groups.

* NOTE: the ‘target’ column indicates a successful transaction (‘1’) or a no-transaction (‘0’). Verify these are the only values in that column.

* Check the correlation values between all predictor columns to ensure there are no substantial correlations between predictors. This is important to support the decision to classify the ‘target’ using Naïve Bayes.

* Create two data frames: one with all successful transactions, one with all unsuccessful transactions. Make sure they are copies and not slices.

# Data Processing

* Create two data frames: one with all the predictor columns (everything except for Unnamed: 0, ID_code and target) and one with just the target. Make sure they are copies and not slices.

* Define a Gaussian Naïve Bayes model using Sklearn.

* Divide the two data frames you created in step #10 into training and testing subsets.

* Train the model using the training subset of the dataset.

* Test the model using the testing subset of the dataset. Calculate and report the accuracy.

* Perform a cross-validation loop to calculate the accuracy of your model. Report that accuracy. How does it compare to the accuracy you calculated in #14?

* Plot a histogram of the accuracy scores you generated in your cross-validation loop. What do you notice about the distribution of accuracy scores?

* Present the confusion matrix and the results of your Classification Report (sklearn.metrics.classification_report). What do you notice?

* The training data is very skewed towards non-successful transactions (about 90% of the training data has ‘target’==0). Remove enough non-successful transaction rows so that your remaining training data is 50%/50% split between successful and non-successful transactions. Hint: you can use the data frames you created in step #9.

* Repeat the cross-validation process on this data set. Report what your cross-validation accuracy is in this 50/50 case.

# Data Visualization

* Compare the results of your cross-validation with the whole training data and the reduced 50/50 training data

* Present the confusion matrix and the results of your Classification Report (sklearn.metrics.classification_report)

# Communication of Results

* Communicate the results of your analysis.